# Chatbot RAG usando LLM Destilado y Embebidos Locales

En este notebook leeremos documentos desde una carpeta seleccionada (`data/`), usaremos embeddings 100% locales con `sentence-transformers`, los almacenaremos en FAISS, y generaremos un chatbot que responda en base a esa información con nuestro TinyLlama.

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFDirectoryLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("Cargando archivos TXT de la carpeta 'data/'...")
loader_txt = DirectoryLoader('data/', glob="**/*.txt", loader_cls=TextLoader)
documentos_txt = loader_txt.load()

print("Scrapeando datos abiertos del repositorio de la Universidad Nacional de Colombia...")
# URL publica de ejemplo en el Repositorio Institucional UN
url_repositorio = "https://repositorio.unal.edu.co/handle/10336/1"
loader_web = WebBaseLoader(url_repositorio)
documentos_web = loader_web.load()
print(f"Se obtuvieron {len(documentos_web)} documentos de la web.")

# Consolidar documentos
documentos = documentos_txt + documentos_web
print(f"En total, se tienen {len(documentos)} documentos para procesar exitosamente.")

Cargando archivos TXT de la carpeta 'data/'...
Scrapeando datos abiertos del repositorio de la Universidad Nacional de Colombia...
Se obtuvieron 1 documentos de la web.
En total, se tienen 1 documentos para procesar exitosamente.


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
textos_divididos = text_splitter.split_documents(documentos)
print(f"El texto fue subdividido en {len(textos_divididos)} fragmentos.")

El texto fue subdividido en 6 fragmentos.


In [4]:
# Embebidos 100% locales con all-MiniLM-L6-v2 (muy eficiente)
print("Descargando/Cargando modelo de embeddings en HuggingFace...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Creando y llenando la base de datos vectorial local (FAISS)...")
vectorstore = FAISS.from_documents(textos_divididos, embeddings)
print("Base creada correctamente!")

Descargando/Cargando modelo de embeddings en HuggingFace...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creando y llenando la base de datos vectorial local (FAISS)...
Base creada correctamente!


In [5]:
# Cargamos nuevamente el modelo LLM ligero desde HF.
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"
model_path = hf_hub_download(repo_id=model_name_or_path, filename=model_basename)

llm = LlamaCpp(
    model_path=model_path,
    temperature=0.1,  # Temperatura baja para evitar alucinación RAG
    max_tokens=256,
    top_p=1,
    verbose=False  # Para no llenar de logs la ejecución
)

In [6]:
from langchain.prompts import PromptTemplate

prompt_template = """<|system|>
Usa el siguiente contexto para responder a la pregunta del usuario. Si no sabes la respuesta, di simplemente que no lo sabes.
{context}
<|user|>
{question}
<|assistant|>"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

In [7]:
pregunta = "¿Cuantas neuronas tiene el primer ejemplo de redes neuronales?"
print(f"Analizando pregunta: {pregunta}...")
respuesta = qa_chain.invoke({"query": pregunta})

print("\n--- Respuesta del LLM Rag ---")
print(respuesta['result'])

print("\n--- Fuentes de datos usadas ---")
for doc in respuesta['source_documents']:
    print(f"-> {doc.metadata['source']}:\n {doc.page_content}")

Analizando pregunta: ¿Cuantas neuronas tiene el primer ejemplo de redes neuronales?...

--- Respuesta del LLM Rag ---

La pregunta se refiere al número de neuronas en un ejemplo específico de redes neuronales. Sin embargo, no se proporciona un ejemplo específico en la pregunta. Por lo tanto, no se puede responder con precisión a la pregunta actual.

Si se proporciona un ejemplo específico de red neuronal en una pregunta futura, se puede intentar responder a la pregunta sobre el número de neuronas en ese ejemplo específico.

Pero en este momento, sin un ejemplo específico de red neuronal proporcionado en la pregunta, no se puede responder con precisión a la pregunta actual sobre cuántas neuronas tiene el primer ejemplo de redes neuronales.

--- Fuentes de datos usadas ---
-> https://repositorio.unal.edu.co/handle/10336/1:
 de privacidadAcuerdo de usuario finalEnviar Sugerencias
-> https://repositorio.unal.edu.co/handle/10336/1:
 a la página de inicio Nivel nacional  Amazonía  Bogotá  Ca

In [8]:
import gradio as gr

def responder_rag(mensaje, historial):
    try:
        # Tomar los últimos 3 mensajes del historial
        historial_reciente = historial[-3:]
        contexto = ""
        if historial_reciente:
            contexto = "Contexto de conversacion previa:\n"
            for usr, bot in historial_reciente:
                contexto += f"Usuario: {usr}\nAsistente: {bot}\n\n"
            contexto += "Teniendo en cuenta el historial anterior, responde la nueva pregunta: "
            
        pregunta_final = contexto + mensaje
        
        result = qa_chain.invoke({"query": pregunta_final})
        respuesta = result['result']
        fuentes = result.get('source_documents', [])
        
        texto_fuentes = "\n\n**Fuentes consultadas:**"
        if fuentes:
            nombres_archivos = set([doc.metadata.get('source', 'Desconocido') for doc in fuentes])
            for nombre in nombres_archivos:
                texto_fuentes += f"\n- {nombre}"
        else:
            texto_fuentes += "\nNo se usaron fuentes específicas."
            
        return respuesta + texto_fuentes
    except Exception as e:
        return f"Ocurrió un error al procesar la respuesta: {e}"

demo_rag = gr.ChatInterface(
    fn=responder_rag,
    title="Chatbot RAG con Memoria",
    description="Este chatbot RAG recuerda tus últimos 3 mensajes para mantener el contexto."
)

demo_rag.launch(share=True)

Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 4.25.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://d3830782e232ed1d61.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
